In [3]:
# -*- coding: utf-8 -*-
import os
import json
import time
import numpy as np
import tensorflow as tf

# ==========================================
# KONFIGURASI PATH
# ==========================================
# Arahkan ke file TFLite asli MCU-Quake
TFLITE_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20/lite_model.tflite'
# Path ke direktori embedding KDE Indonesia Anda
EMB_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon/output/indonesia_domain_embeddings_3c_le"
EMBEDDING_DIM = 32

def main():
    print("="*60)
    print("BUILD & PROFILING: MCU-QUAKE (KOMPONEN Z) DENGAN EMBEDDING INDONESIA")
    print("="*60)

    # 1. Analisis Model TFLite Asli (Feature Extractor)
    if os.path.exists(TFLITE_PATH):
        flash_size_kb = os.path.getsize(TFLITE_PATH) / 1024.0
        
        interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
        interpreter.allocate_tensors()
        
        tensor_details = interpreter.get_tensor_details()
        ram_bytes = sum([np.prod(t['shape']) * np.dtype(t['dtype']).itemsize for t in tensor_details if t['shape'] is not None])
        
        # Uji latensi inferensi dummy
        input_details = interpreter.get_input_details()
        input_shape = input_details[0]['shape']
        dummy_input = np.random.randn(*input_shape).astype(np.float32)
        
        interpreter.set_tensor(input_details[0]['index'], dummy_input)
        start_time = time.time()
        for _ in range(100):
            interpreter.set_tensor(input_details[0]['index'], dummy_input)
            interpreter.invoke()
        latency_ms = ((time.time() - start_time) / 100) * 1000

        print(f"\n[A] Analisis Feature Extractor (TFLite):")
        print(f"    - Ukuran Flash (ROM) : {flash_size_kb:.2f} KB")
        print(f"    - Estimasi RAM       : {ram_bytes / 1024:.2f} KB")
        print(f"    - Latensi Rata-rata  : {latency_ms:.2f} ms (di PC)")
    else:
        print(f"\n[!] File TFLite tidak ditemukan di: {TFLITE_PATH}")
        print("    Pastikan Anda mengarahkannya ke file .tflite bawaan repositori.")

    # 2. Analisis Beban Memori KDE Indonesia (Khusus Komponen Z)
    print(f"\n[B] Analisis Ruang Probabilitas (KDE Komponen Z):")
    file_path = os.path.join(EMB_DIR, "Embedding data, Z.json")
    
    total_vectors = 0
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            data = json.load(f)
        n_count = len(data.get("noise", []))
        le_count = len(data.get("le", []))
        total_vectors = n_count + le_count
        print(f"    - Komponen Z: {total_vectors} vektor ({n_count} Noise, {le_count} LE)")
    else:
        print(f"    - Komponen Z: File JSON tidak ditemukan di direktori!")

    # Kalkulasi memori KDE dengan asumsi kuantisasi INT8 (1 Byte per parameter)
    kde_ram_kb = (total_vectors * EMBEDDING_DIM * 1) / 1024.0

    print(f"\n    > Total Vektor Laten Komponen Z : {total_vectors} titik")
    print(f"    > Beban Memori Tambahan (INT8)  : {kde_ram_kb:.2f} KB")

    print("\n" + "="*60)
    print("RINGKASAN KELAYAKAN TINYML (DEPLOYMENT SUMMARY)")
    print("="*60)
    print(f"Model dengan Komponen Z tunggal siap di-deploy ke ESP32/STM32.")
    print(f"Footprint memori sistem terintegrasi menjadi sangat ramping (~{kde_ram_kb:.2f} KB untuk KDE).")
    print("="*60)

if __name__ == "__main__":
    main()

BUILD & PROFILING: MCU-QUAKE (KOMPONEN Z) DENGAN EMBEDDING INDONESIA

[A] Analisis Feature Extractor (TFLite):
    - Ukuran Flash (ROM) : 16.16 KB
    - Estimasi RAM       : 84.38 KB
    - Latensi Rata-rata  : 0.01 ms (di PC)

[B] Analisis Ruang Probabilitas (KDE Komponen Z):
    - Komponen Z: 10318 vektor (5159 Noise, 5159 LE)

    > Total Vektor Laten Komponen Z : 10318 titik
    > Beban Memori Tambahan (INT8)  : 322.44 KB

RINGKASAN KELAYAKAN TINYML (DEPLOYMENT SUMMARY)
Model dengan Komponen Z tunggal siap di-deploy ke ESP32/STM32.
Footprint memori sistem terintegrasi menjadi sangat ramping (~322.44 KB untuk KDE).


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [7]:
# -*- coding: utf-8 -*-
import os
import json

# ==========================================
# KONFIGURASI PATH
# ==========================================
TFLITE_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20/lite_model.tflite'
EMB_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon/output/indonesia_domain_embeddings_3c_le"

# Direktori Output untuk file C++
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models'

def ensure_dir():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

def export_tflite_to_c_array():
    print("\n[1] Mengekspor Model TFLite ke C-Array...")
    if not os.path.exists(TFLITE_PATH):
        print(f"  ❌ File tidak ditemukan: {TFLITE_PATH}")
        return

    with open(TFLITE_PATH, "rb") as f:
        tflite_data = f.read()

    hex_array = [f"0x{b:02x}" for b in tflite_data]
    c_code = "#ifndef MCU_QUAKE_MODEL_H\n#define MCU_QUAKE_MODEL_H\n\n"
    c_code += "// File TFLite Feature Extractor MCU-Quake\n"
    c_code += f"const unsigned int mcu_quake_model_len = {len(tflite_data)};\n"
    c_code += "const unsigned char mcu_quake_model[] = {\n    "
    
    for i in range(0, len(hex_array), 12):
        c_code += ", ".join(hex_array[i:i+12]) + ",\n    "
    c_code = c_code.rstrip(",\n    ") + "\n};\n\n#endif // MCU_QUAKE_MODEL_H"

    out_path = os.path.join(OUTPUT_DIR, "mcu_quake_model.h")
    with open(out_path, "w") as f:
        f.write(c_code)
    print(f"  ✅ Tersimpan: {out_path} ({len(tflite_data)/1024:.2f} KB)")

def export_kde_z_to_c_array():
    print("\n[2] Mengekspor Matriks KDE Komponen Z ke C-Array...")
    json_path = os.path.join(EMB_DIR, "Embedding data, Z.json")
    
    if not os.path.exists(json_path):
        print(f"  ❌ File tidak ditemukan: {json_path}")
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    noise_vecs = data.get("noise", [])
    le_vecs = data.get("le", [])

    if len(noise_vecs) == 0:
        print("  ❌ Data vektor kosong.")
        return

    # DETEKSI OTOMATIS: Mengecek apakah ini array 1D atau 2D
    first_item = noise_vecs[0]
    is_2d = isinstance(first_item, (list, tuple))
    
    c_code = "#ifndef KDE_Z_VECTORS_H\n#define KDE_Z_VECTORS_H\n\n"
    c_code += "// Ruang Probabilitas Lokal Indonesia (Komponen Z)\n"
    
    # --- FUNGSI PEMBANTU UNTUK FORMATTING ---
    def format_array(name, vectors):
        code = f"const int NUM_{name.upper()} = {len(vectors)};\n"
        
        if is_2d: # Jika Nested Array (Misal: [[0.1, 0.2], [0.3, 0.4]])
            dim = len(vectors[0])
            code += f"const float kde_{name.lower()}_vectors[][{dim}] = {{\n"
            for vec in vectors:
                formatted_vec = ", ".join([f"{val:.6f}" for val in vec])
                code += f"    {{{formatted_vec}}},\n"
            code = code.rstrip(",\n") + "\n};\n\n"
            
        else: # Jika Flat Array 1D (Misal: [0.1, 0.2, 0.3, 0.4])
            code += f"const float kde_{name.lower()}_vectors[] = {{\n    "
            formatted_vecs = [f"{val:.6f}" for val in vectors]
            for i in range(0, len(formatted_vecs), 12):
                code += ", ".join(formatted_vecs[i:i+12]) + ",\n    "
            code = code.rstrip(",\n    ") + "\n};\n\n"
            
        return code

    # Menjalankan formatting untuk derau (Noise) dan gempa (LE)
    c_code += format_array("NOISE", noise_vecs)
    c_code += format_array("LE", le_vecs)
    c_code += "#endif // KDE_Z_VECTORS_H"

    out_path = os.path.join(OUTPUT_DIR, "kde_z_vectors.h")
    with open(out_path, "w") as f:
        f.write(c_code)
    
    bentuk_array = "2D (Nested Array)" if is_2d else "1D (Flat Array)"
    print(f"  ✅ Tersimpan: {out_path} (Terdeteksi Format: {bentuk_array})")

def main():
    print("="*60)
    print("GENERATE C-ARRAY UNTUK ESP32-S3 DEPLOYMENT")
    print("="*60)
    ensure_dir()
    export_tflite_to_c_array()
    export_kde_z_to_c_array()
    print("="*60)
    print("Proses selesai. Pindahkan file .h di dalam folder 'esp32_deployment_files'")
    print("ke dalam folder proyek C++ / Arduino IDE Anda.")
    print("="*60)

if __name__ == "__main__":
    main()

GENERATE C-ARRAY UNTUK ESP32-S3 DEPLOYMENT

[1] Mengekspor Model TFLite ke C-Array...
  ✅ Tersimpan: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/mcu_quake_model.h (16.16 KB)

[2] Mengekspor Matriks KDE Komponen Z ke C-Array...
  ✅ Tersimpan: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/kde_z_vectors.h (Terdeteksi Format: 1D (Flat Array))
Proses selesai. Pindahkan file .h di dalam folder 'esp32_deployment_files'
ke dalam folder proyek C++ / Arduino IDE Anda.


In [10]:
# -*- coding: utf-8 -*-
import json
import os

# ==========================================
# KONFIGURASI PATH
# ==========================================
DATA_JSON_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "test_samples.h")

def main():
    # Memastikan folder output tersedia
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"📁 Membuat folder baru: {OUTPUT_DIR}")

    print("Membaca dataset pengujian...")
    with open(DATA_JSON_PATH, "r") as f:
        test_data = json.load(f)
    
    keys = list(test_data.keys())
    
    # Ambil gelombang Z (700 sampel) dari sampel pertama
    sample_eq = test_data[keys[0]]["Z"][:700]        # Sinyal Gempa
    sample_noise = test_data[keys[0]]["Z_noise"][-700:] # Sinyal Derau
    
    c_code = "#ifndef TEST_SAMPLES_H\n#define TEST_SAMPLES_H\n\n"
    c_code += "// Sampel Data Pengujian Langsung dari JSON\n\n"
    
    # Format Gempa
    c_code += "const float test_wave_earthquake[700] = {\n    "
    c_code += ", ".join([f"{val:.6f}" for val in sample_eq])
    c_code += "\n};\n\n"
    
    # Format Derau
    c_code += "const float test_wave_noise[700] = {\n    "
    c_code += ", ".join([f"{val:.6f}" for val in sample_noise])
    c_code += "\n};\n\n"
    
    c_code += "#endif // TEST_SAMPLES_H"
    
    with open(OUTPUT_FILE, "w") as f:
        f.write(c_code)
        
    print(f"✅ Berhasil mengekstrak gelombang uji ke: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Membaca dataset pengujian...
✅ Berhasil mengekstrak gelombang uji ke: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/test_samples.h
